# 11_02 · 정리 데이터 풀런 오류 분석 (`11_01`)

- **비교 기준 = exp2 focal(len512)의 로짓을 정리 test 행으로 정렬한 것.** `11_01`은 정리 test(11,244)에서 평가됐으므로 구 test(11,271) 수치와 직접 대면하지 않는다(ADR-0010).
- **판정 축**: 이 런은 데이터 클리닝과 신 레시피(batch, lr 변화)가 동시에 바뀌어 두 성분이 분리되지 않는다. 라벨 충돌 관련 74클래스 대 비관련 114클래스의 per-class F1 paired 대조로 데이터 클리닝 영향을 평가할 수 있다. 클리닝이 겨냥한 대상은 연루 클래스이고 레시피 변화는 두 집단에 함께 작용하므로, 두 집단의 ΔF1 차이가 클리닝 성분의 추정치다.
- 임계: `sigmoid τ=0.5 ⟺ logit≥0`.
- 오류 분석 기법은 모듈의 `TECHNIQUES` 레지스트리로 관리한다 — 기법 추가 = 함수 작성 + 레지스트리 한 줄이면 표·저장에 자동 반영.

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

from huggingface_hub import hf_hub_download
from datasets import load_dataset

# 오류 분석 하니스(계산 로직) — src/error_analysis.py
from error_analysis import ErrorAnalysis
# 훈련 하니스와 같은 지표 정의를 쓴다(τ 규약·F1 3종·empty rate의 SSOT)
from patent_train.metrics import f1_triple, empty_rate, sigmoid

In [2]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "split": "test",
    "tau": 0.5,                 # sigmoid τ=0.5 ⟺ logit≥0 = focal native 임계
    "raw_ds": "ingyoun/patent-clean-text",       # 정리본(10_03 반영) — test 11,244
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": str(HF_HOME),
    "out_path": ROOT / "output",
}

# 대상 = 11_01 정리 데이터 풀런. 기준 = exp2 focal(구 test 로짓을 정리 test 행으로 정렬).
TARGET = {"tag": "modernbert-patent-len512-b128", "arch": "modernbert", "recipe": "eff128/lr4.8e-4", "data": "clean"}
REF_SRC = "modernbert-patent-len512"             # 구 test(11,271) 로짓 캐시의 tag
REF_TAG = "modernbert-patent-len512@clean"       # 정렬 후 tag(파일이 아닌 배열로 등록)

In [3]:
def load_ssot(tag):
    """
    test SSOT를 스키마 무관하게 정규화(micro/macro/sample/empty_rate)해 반환.
    - src 훈련 하니스: {tag}_metrics.json          (split 중첩: {"test": {"test_micro_f1", …}})
    - 손실 실험     : {tag}_test_metrics.json      (flat: test_micro_f1 …)
    - focal/baseline: total_metrics_{tag}.json     (rich: multilabel_f1.keep · empty_rate_tau_micro)
    셋 다 없으면 None.
    """
    def flatten(d):
        return {"micro": d["test_micro_f1"], "macro": d["test_macro_f1"],
                "sample": d["test_sample_f1"], "empty_rate": d["test_empty_rate"]}

    nested = config["out_path"] / f"{tag}_metrics.json"
    flat = config["out_path"] / f"{tag}_test_metrics.json"
    rich = config["out_path"] / f"total_metrics_{tag}.json"
    if nested.exists():
        return flatten(json.loads(nested.read_text(encoding="utf-8"))["test"])
    if flat.exists():
        return flatten(json.loads(flat.read_text(encoding="utf-8")))
    if rich.exists():
        d = json.loads(rich.read_text(encoding="utf-8"))
        return {**d["multilabel_f1"]["keep"], "empty_rate": d["empty_rate_tau_micro"]}
    return None

In [4]:
random.seed(config['seed'])
np.random.seed(config['seed'])

In [5]:
path = hf_hub_download(
    repo_id=config["raw_ds"],
    filename="label_mappings.json",
    repo_type="dataset",
)

with open(path, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

In [6]:
EA = ErrorAnalysis(label_mapping, num_labels=config["num_labels"], tau=config["tau"])
LS = EA.ls   # 표·저장 셀에서 참조(Lno 축)
print(f"C={LS.C}  L={LS.L}  Lno={LS.lnos}")

C=188  L=17  Lno=['EA', 'EB', 'EC', 'ED', 'EE', 'EF', 'EG', 'EH', 'EI', 'LA', 'LB', 'LC', 'NB', 'NC', 'ND', 'OA', 'OB']


## 데이터 · 정리 test 행 정렬

정리본 test(11,244)가 평가 축이다. `11_01` 로짓은 이 축에 그대로 얹히지만, exp2 로짓은 구 test(11,271) 축이라 **공통 문서 행만 남겨 정렬**한다 — `doc_ids_test.json`(구 순서)에서 정리본 `document_id`의 위치를 찾아 행을 고른다. 두 로짓이 같은 문서·같은 순서를 보는지 확인하고, 빠진 문서 집합이 `10_03`의 제거 목록(test 성분)과 정확히 일치하는지 대조한다.

In [7]:
ds = load_dataset(config["raw_ds"], split=config["split"])

EA.set_data(ds)
Y, length_bin, k_gold, BINS = EA.Y, EA.length_bin, EA.k_gold, EA.bins   # 표·verify 셀에서 참조
N = EA.n

print(f"N={N:,} || 라벨 개수 : {np.bincount(k_gold)[1:6].tolist()}(k=1 ~ 5) ||  k>=2 비율 {(k_gold >= 2).mean():.2%}")
print("길이 bin " + "  || ".join(f"{b} : {int((length_bin == b).sum()):,}" for b in BINS))

N=11,244 || 라벨 개수 : [9555, 1275, 298, 67, 33](k=1 ~ 5) ||  k>=2 비율 15.02%
길이 bin <=512 : 3,187  || 512-1024 : 5,172  || 1024-2048 : 2,336  || >2048 : 549


In [8]:
# 두 로짓 세트의 행 축은 분리돼 있다:
#   doc_ids_{split}.json       — 구 로짓(exp2·KoBERT·손실 3종, 11,271행) 행 축(과거 코드 재현용 유지)
#   doc_ids_clean_{split}.json — 정리 로짓(b128 등, 11,244행) 행 축 = 정리 데이터셋 document_id 순서
# b128은 clean 축 그대로, exp2는 구 축을 clean 축으로 사영(KEEP)해 공통 축에 얹는다.
old_ids = json.loads((config["out_path"] / f"doc_ids_{config['split']}.json").read_text(encoding="utf-8"))
clean_ids = json.loads((config["out_path"] / f"doc_ids_clean_{config['split']}.json").read_text(encoding="utf-8"))
new_ids = list(ds["document_id"])
pos = {d: i for i, d in enumerate(old_ids)}
KEEP = np.array([pos[d] for d in new_ids])          # (N,) 정리 test i번 문서의 구 test 행 번호

print(f"구 test {len(old_ids):,} → 정리 test {len(new_ids):,} (제거 {len(old_ids) - len(new_ids)})")

구 test 11,271 → 정리 test 11,244 (제거 27)


In [9]:
### verify — clean 축이 데이터셋과 일치 · 정렬 축이 10_03 제거 목록과 일치
assert clean_ids == new_ids, "doc_ids_clean이 정리 데이터셋 document_id 순서와 다르다(b128 로짓 축)"
assert len(KEEP) == N and np.all(np.diff(KEEP) > 0), "정렬 인덱스가 단조 증가가 아니다(행 순서 불일치)"

rec_clean = json.loads((config["out_path"] / "label_conflict_docs.json").read_text(encoding="utf-8"))
removed = set(old_ids) - set(new_ids)
expected = set(rec_clean["conflict_docs"] + rec_clean["eval_leak_docs"] + rec_clean["valtest_dedup_docs"]) & set(old_ids)
assert removed == expected, (len(removed), len(expected))
print(f"verify(축) pass — clean 축==데이터셋 · 행 순서 보존 · 제거 {len(removed)}건이 10_03 제거 목록의 test 성분과 일치")

verify(축) pass — clean 축==데이터셋 · 행 순서 보존 · 제거 27건이 10_03 제거 목록의 test 성분과 일치


## Logit

In [10]:
EA.add([TARGET], cache_dir=config["out_path"], split=config["split"])        # 파일 → 로드·분석

ref_full = np.load(config["out_path"] / f"logits_{REF_SRC}_{config['split']}.npy")
EA.add_logits(REF_TAG, ref_full[KEEP])                                       # 정렬 배열 → 분석

results, records = EA.models, EA.records
TAGS = [REF_TAG, TARGET["tag"]]     # 표 출력 순서: 기준 → 대상
print(list(results))

[load] logits_modernbert-patent-len512-b128_test.npy
['modernbert-patent-len512-b128', 'modernbert-patent-len512@clean']


## 헤드라인 — 정리 test 공통 축

In [11]:
pred = {tag: (m.P >= EA.tau) for tag, m in results.items()}
head = {tag: {**f1_triple(Y, p), "empty_rate": empty_rate(p),
              "p@1": records[tag]["anchor_error"]["p@1"]} for tag, p in pred.items()}

print(f"{'tag':<34}{'micro':>9}{'macro':>9}{'sample':>9}{'empty':>9}{'P@1':>9}")
for tag in TAGS:
    h = head[tag]
    print(f"{tag:<34}{h['micro_f1']:>9.4f}{h['macro_f1']:>9.4f}{h['sample_f1']:>9.4f}"
          f"{h['empty_rate']:>9.2%}{h['p@1']:>9.4f}")

d = {k: head[TARGET["tag"]][k] - head[REF_TAG][k] for k in ("micro_f1", "macro_f1", "sample_f1", "empty_rate", "p@1")}
print(f"{'Δ(11_01 - exp2, pt)':<34}{100*d['micro_f1']:>+9.2f}{100*d['macro_f1']:>+9.2f}"
      f"{100*d['sample_f1']:>+9.2f}{100*d['empty_rate']:>+9.2f}{100*d['p@1']:>+9.2f}")

tag                                   micro    macro   sample    empty      P@1
modernbert-patent-len512@clean       0.8599   0.8571   0.8718    1.80%   0.8998
modernbert-patent-len512-b128        0.8588   0.8565   0.8738    1.17%   0.9000
Δ(11_01 - exp2, pt)                   -0.12    -0.06    +0.20    -0.62    +0.02


In [12]:
### verify — 두 모델의 헤드라인이 각자의 SSOT와 일치
# 11_01 SSOT는 group_by_length 로더, 재덤프 로짓은 순차 로더 — 배치 패딩 폭이 달라 bf16 누적
# 순서가 미세하게 바뀌므로 b128은 1e-3으로 대조한다(순위·결론 불변). exp2 정렬본은 exact.
s = load_ssot(TARGET["tag"])                       # 11_01: runner.save_metrics 산출({tag}_metrics.json)
for k, v in [("micro_f1", "micro"), ("macro_f1", "macro"), ("sample_f1", "sample"), ("empty_rate", "empty_rate")]:
    assert abs(head[TARGET["tag"]][k] - s[v]) < 1e-3, (k, head[TARGET["tag"]][k], s[v])

# exp2 정렬본: 10_03이 같은 방식(제거 행 제외 재계산)으로 낸 정리 test 재계산값
hl = json.loads((config["out_path"] / "headline_cleaned_test.json").read_text(encoding="utf-8"))["models"][REF_SRC]["new"]
assert hl["micro"] == round(head[REF_TAG]["micro_f1"], 4), (hl["micro"], head[REF_TAG]["micro_f1"])
assert hl["p_at_1"] == round(head[REF_TAG]["p@1"], 4), (hl["p_at_1"], head[REF_TAG]["p@1"])
print(f"verify(헤드라인) pass — 11_01 == {TARGET['tag']}_metrics.json · exp2 정렬본 == headline_cleaned_test.json[new]")

verify(헤드라인) pass — 11_01 == modernbert-patent-len512-b128_metrics.json · exp2 정렬본 == headline_cleaned_test.json[new]


## 앵커(top-1) 오류 분해 — sibling vs cross-Lno

In [13]:
anchor = {tag: records[tag]["anchor_error"] for tag in TAGS}

header = (f"{'tag':<34}{'P@1':>8}{'오류':>8}"
          f"{'sibling':>14}{'cross-Lno':>16}{'우연':>6}{'배수':>8}")
print(header)
print("-" * len(header))

for tag, r in anchor.items():
    sib = f"{r['sibling']:,} ({r['sibling_ratio']:.1%})"
    cro = f"{r['cross_lno']:,} ({1 - r['sibling_ratio']:.1%})"
    print(f"{tag:<34}{r['p@1']:>8.4f}{r['n_error']:>8,}"
          f"{sib:>16}{cro:>16}"
          f"{r['chance_sibling_ratio']:>8.1%}{r['sibling_enrichment']:>7.1f}x")

tag                                    P@1      오류       sibling       cross-Lno    우연      배수
----------------------------------------------------------------------------------------------
modernbert-patent-len512@clean      0.8998   1,127     391 (34.7%)     736 (65.3%)    7.0%    5.0x
modernbert-patent-len512-b128       0.9000   1,124     405 (36.0%)     719 (64.0%)    6.9%    5.2x


## 멀티라벨(τ=0.5) 오류 분해 — FP · FN

손실 축 종결(ADR-0009)의 기제는 FP:FN 부호였다 — focal 대비 손실 변형이 k=1을 과대예측하거나 k≥2를 과소예측했다. 레시피가 바뀐 focal 런이 이 균형을 유지하는지 본다.

In [14]:
multilabel = {tag: records[tag]["multilabel_error"] for tag in TAGS}
print(f"{'tag':<34}{'FP':>8}{'FP sib':>16}{'FN':>8}{'FN sib':>16}{'empty':>11}")

for tag, r in multilabel.items():
    print(f"{tag:<34}{r['fp']:>8,}{r['fp_sibling']:>9,} ({r['fp_sibling_ratio']:>5.1%})"
          f"{r['fn']:>8,}{r['fn_sibling']:>9,} ({r['fn_sibling_ratio']:>5.1%}){r['empty_rate']:>9.2%}")

tag                                     FP          FP sib      FN          FN sib      empty
modernbert-patent-len512@clean       1,915      760 (39.7%)   1,881      661 (35.1%)    1.80%
modernbert-patent-len512-b128        2,004      793 (39.6%)   1,842      694 (37.7%)    1.17%


## 라벨 개수 bin — 단일(k=1) vs 다라벨(k≥2) · 카디널리티 헤드룸

주 지표는 micro-F1이므로 k≥2의 무게를 문서 비율이 아니라 **양성 라벨 인스턴스 비율**로 환산하고, 결손이 표현(랭킹)에서 오는지 결정 규칙(카디널리티)에서 오는지 가른다 — k≥2 문서에만 **오라클 카디널리티**(정답 개수 `k`를 알고 상위 `k`개 선택, 랭킹 불변)를 적용한 micro-F1이 도달 불가 상한이다.

In [15]:
count_bin = {tag: records[tag]["label_count_bins"] for tag in TAGS}

print(f"{'tag':<34}{'bin':>4}{'n':>8}{'micro':>9}{'sample':>9}{'R-Prec':>9}{'FP':>8}{'FN':>8}{'FP:FN':>8}")
for tag, r in count_bin.items():
    for name in ["k=1", "k>=2"]:
        v = r[name]
        print(f"{tag:<34}{name:>4}{v['n']:>8,}{v['micro_f1']:>9.4f}{v['sample_f1']:>9.4f}"
              f"{v['r_precision']:>9.4f}{v['fp']:>8,}{v['fn']:>8,}{v['fp_fn_ratio']:>8.2f}")
    gap = r["k=1"]["sample_f1"] - r["k>=2"]["sample_f1"]
    print(f"{'':<34}{'[sample f1(k=1) - sample f1(k>=2) : ':>7}{'':>8}{'':>9}{gap:>9.4f}]\n")

tag                                bin       n    micro   sample   R-Prec      FP      FN   FP:FN
modernbert-patent-len512@clean     k=1   9,555   0.8820   0.8875   0.8942   1,548     796    1.94
modernbert-patent-len512@clean    k>=2   1,689   0.7994   0.7829   0.8617     367   1,085    0.34
                                  [sample f1(k=1) - sample f1(k>=2) :                     0.1046]

modernbert-patent-len512-b128      k=1   9,555   0.8814   0.8906   0.8947   1,628     743    2.19
modernbert-patent-len512-b128     k>=2   1,689   0.7961   0.7787   0.8593     376   1,099    0.34
                                  [sample f1(k=1) - sample f1(k>=2) :                     0.1119]



In [16]:
cardinality = {tag: records[tag]["cardinality"] for tag in TAGS}

print(f"양성 라벨 인스턴스 {int(Y.sum()):,} · k≥2 점유율 {cardinality[TAGS[0]]['pos_share_k>=2']:.1%}\n")
print(f"{'tag':<34}{'micro':>9}{'+오라클k':>11}{'이득':>9}{'k≥2과소예측':>13}{'예측/정답':>12}")
for tag, r in cardinality.items():
    kp = f"{r['mean_k_pred_k>=2']:.2f}/{r['mean_k_gold_k>=2']:.2f}"
    print(f"{tag:<34}{r['micro']:>9.4f}{r['micro_oracle_k_on_multi']:>12.4f}"
          f"{r['oracle_k_gain_pt']:>+12.2f}{r['under_predict_rate_k>=2']:>15.1%}{kp:>16}")

양성 라벨 인스턴스 13,534 · k≥2 점유율 29.4%

tag                                   micro      +오라클k       이득      k≥2과소예측       예측/정답
modernbert-patent-len512@clean       0.8599      0.8758       +1.59          44.8%       1.93/2.36
modernbert-patent-len512-b128        0.8588      0.8751       +1.64          44.7%       1.93/2.36


## 길이 bin × 오류 유형

In [17]:
bin_error = {tag: records[tag]["length_bin_error"] for tag in TAGS}

for tag, r in bin_error.items():
    print(tag)
    print(f"  {'bin':<12}{'n':>7}{'오류율':>8}{'sibling비':>10}{'FP/문서':>9}{'FN/문서':>8}")
    for b in BINS:
        v = r[b]
        print(f"  {b:<12}{v['n']:>7,}{v['anchor_error_rate']:>9.2%}{v['sibling_ratio']:>11.1%}"
              f"{v['fp_per_doc']:>10.3f}{v['fn_per_doc']:>10.3f}")
    print()

modernbert-patent-len512@clean
  bin               n     오류율  sibling비    FP/문서   FN/문서
  <=512         3,187    8.41%      34.7%     0.155     0.149
  512-1024      5,172    9.67%      33.4%     0.166     0.161
  1024-2048     2,336   11.77%      35.3%     0.191     0.197
  >2048           549   15.30%      40.5%     0.220     0.209

modernbert-patent-len512-b128
  bin               n     오류율  sibling비    FP/문서   FN/문서
  <=512         3,187    8.60%      38.0%     0.161     0.142
  512-1024      5,172    9.40%      34.4%     0.175     0.158
  1024-2048     2,336   11.94%      38.0%     0.200     0.197
  >2048           549   15.48%      32.9%     0.211     0.197



## 클리닝 성분 분리 — 라벨 충동 관련 클래스 per-class paired 대조

클리닝이 손댄 대상은 충돌 연루 클래스(74개, `per_class_f1_modern512.json`의 `conflict_groups>0`)이고 레시피 변화는 두 집단에 함께 작용하므로, 연루·비연루 두 집단의 ΔF1 차이를 클리닝 성분의 추정치로 읽는다.

- 귀무가설: 클리닝이 연루 클래스에 특별한 효과가 없다 → 두 집단의 ΔF1 분포가 같다.
- 검정: 클래스를 집단 라벨에 대해 무작위 재배치하는 순열검정(10,000회). 클래스별 ΔF1이 표본 단위이며 두 모델이 같은 test 문서를 봤으므로 paired다.
- 이득 방향이 아니라 분리 가능성을 재는 절이다 — 유의하지 않으면 "단일 런에서 클리닝 성분이 노이즈와 분리되지 않는다"는 ADR-0010의 예측이 실측으로 확인된 것이다.

In [24]:
PERM = 10_000

pc_ref = json.loads((config["out_path"] / "per_class_f1_modern512.json").read_text(encoding="utf-8"))["per_class"]
involved = {mno for mno, v in pc_ref.items() if v["conflict_groups"] > 0}   # 충돌 연루 클래스(10_03 검출)

a = records[REF_TAG]["per_class_f1"]            # exp2(정리 test 정렬)
b = records[TARGET["tag"]]["per_class_f1"]      # 11_01
MNOS = LS.mno_of_col
delta = np.array([b[m]["f1"] - a[m]["f1"] for m in MNOS])
inv = np.array([m in involved for m in MNOS])

obs = float(delta[inv].mean() - delta[~inv].mean())
rng = np.random.default_rng(config["seed"])
n_inv = int(inv.sum())
perm = np.array([(lambda p: delta[p[:n_inv]].mean() - delta[p[n_inv:]].mean())(rng.permutation(len(delta)))
                 for _ in range(PERM)])
p_value = float((np.abs(perm) >= abs(obs)).mean())

print(f"{'집단':<12}{'n':>5}{'평균 F1(exp2)':>15}{'평균 F1(11_01)':>16}{'평균 ΔF1(pt)':>15}{'개선 클래스':>12}")
for name, sel in [("충돌 연루", inv), ("비연루", ~inv)]:
    fa = np.mean([a[m]["f1"] for m, s in zip(MNOS, sel) if s])
    fb = np.mean([b[m]["f1"] for m, s in zip(MNOS, sel) if s])
    print(f"{name:<12}{int(sel.sum()):>5}{fa:>15.4f}{fb:>16.4f}{100*delta[sel].mean():>+15.2f}"
          f"{f'{(delta[sel] > 0).sum()}/{int(sel.sum())}':>14}")

print(f"\n연루 - 비연루 : {100*obs:+.2f}pt | 순열검정 p={p_value:.3f} ({PERM:,}회)")
print(f"판정: {'연루 클래스에 유의한 초과 이득' if p_value < 0.05 else '단일 런에서 클리닝 성분이 분리되지 않는다(ADR-0010 예측과 일치)'}")

집단              n    평균 F1(exp2)    평균 F1(11_01)     평균 ΔF1(pt)      개선 클래스
충돌 연루          74         0.8313          0.8297          -0.17         31/74
비연루           114         0.8738          0.8739          +0.02        56/114

연루 - 비연루 : -0.18pt | 순열검정 p=0.628 (10,000회)
판정: 단일 런에서 클리닝 성분이 분리되지 않는다(ADR-0010 예측과 일치)


In [19]:
# 연루 클래스 중 충돌 그룹이 많은 상위 클래스 — EB01(7그룹)이 ADR-0010이 지목한 관측 대상
top_conf = sorted(involved, key=lambda m: -pc_ref[m]["conflict_groups"])[:10]
print(f"{'Mno':<8}{'충돌그룹':>8}{'support':>9}{'F1(exp2)':>11}{'F1(11_01)':>11}{'Δ(pt)':>9}")
for m in top_conf:
    print(f"{m:<8}{pc_ref[m]['conflict_groups']:>8}{a[m]['support']:>9}"
          f"{a[m]['f1']:>11.4f}{b[m]['f1']:>11.4f}{100*(b[m]['f1'] - a[m]['f1']):>+9.2f}")

Mno         충돌그룹  support   F1(exp2)  F1(11_01)    Δ(pt)
EB01           7       66     0.8189     0.8571    +3.82
EA11           3       73     0.7467     0.7226    -2.41
EI10           3       74     0.8993     0.8919    -0.74
ND08           2       65     0.7407     0.7571    +1.64
EB05           2       75     0.8859     0.8844    -0.15
EA10           2       78     0.7578     0.7578    +0.00
LA06           2       74     0.8356     0.8310    -0.46
LA01           2       69     0.8788     0.8722    -0.66
EA02           2       69     0.7852     0.8058    +2.06
EB04           2       73     0.9660     0.9595    -0.65


In [20]:
### verify — per-class 산출이 헤드라인과 정합
for tag in TAGS:
    pcf = records[tag]["per_class_f1"]
    assert sum(v["support"] for v in pcf.values()) == int(Y.sum())                     # support 합 == 양성 인스턴스
    macro = float(np.mean([v["f1"] for v in pcf.values()]))
    assert abs(macro - head[tag]["macro_f1"]) < 5e-4, (tag, macro, head[tag]["macro_f1"])   # per-class 평균 == macro
assert len(involved) == 74 and int(inv.sum()) == 74, (len(involved), int(inv.sum()))   # ADR-0010 연루 클래스 수
print("verify(per-class) pass — support 합 == 양성 인스턴스 · per-class 평균 == macro-F1 · 연루 74클래스")

verify(per-class) pass — support 합 == 양성 인스턴스 · per-class 평균 == macro-F1 · 연루 74클래스


## 오류 차집합 — exp2 → 11_01

같은 정리 test 문서 축에서 **어디를 고치고(fixed) 어디를 깨는지(broken)** 본다. 성분은 데이터·레시피가 묶인 채이므로 라벨은 `data+recipe`다. `hard_core`는 두 런이 공통으로 틀린 문서다.

In [21]:
hc = EA.hard_core()                                        # exp2·11_01 공통 오류
cross_model = {
    "pairs": [EA.compare(REF_TAG, TARGET["tag"], "data+recipe")],
    "hard_core": {
        "n": int(hc.sum()),
        "by_length_bin": {b: int((hc & (length_bin == b)).sum()) for b in BINS},
    },
}

for r in cross_model["pairs"]:
    print(f"[{r['component']}] {r['base']} → {r['target']}")
    print(f"  {'':<10}{'n':>7}{'k>=2':>7}{'sibling':>10}{'cross':>8}   " + "".join(f"{b:>12}" for b in BINS))
    for name in ["fixed", "broken"]:
        v = r[name]
        print(f"  {name:<10}{v['n']:>7,}{v['k>=2']:>7,}{v['sibling']:>10,}{v['cross_lno']:>8,}   "
              + "".join(f"{v['by_length_bin'][b]:>12,}" for b in BINS))
    print(f"  {'순이득':<8}{r['net_gain']:>+7,}{'':>25}   "
          + "".join(f"{r['net_by_bin'][b]:>+12,}" for b in BINS))
    print(f"  {'교정률':<8}{'':>7}{'':>25}   "
          + "".join(f"{r['fix_rate_by_bin'][b]:>12.1%}" for b in BINS) + "\n")

print(f"공통 오류(hard core, exp2∩11_01) {cross_model['hard_core']['n']:,}건")
print("  bin별 " + "  ".join(f"{b} {cross_model['hard_core']['by_length_bin'][b]:,}" for b in BINS))

[data+recipe] modernbert-patent-len512@clean → modernbert-patent-len512-b128
                  n   k>=2   sibling   cross          <=512    512-1024   1024-2048       >2048
  fixed         353     45       128     225             94         165          73          21
  broken        350     47       141     209            100         151          77          22
  순이득          +3                                      -6         +14          -4          -1
  교정률                                               35.1%       33.0%       26.6%       25.0%

공통 오류(hard core, exp2∩11_01) 774건
  bin별 <=512 174  512-1024 335  1024-2048 202  >2048 63


## 저장

In [22]:
meta = {
    "split": config["split"],
    "n_docs": int(N),
    "tau": config["tau"],
    "num_labels": config["num_labels"],
    "num_lno": LS.L,
    "fields": config["fields"],
    "raw_ds": config["raw_ds"],
    "ref_tag": REF_TAG,
    "ref_note": f"{REF_SRC}(구 test 11,271) 로짓을 정리 test {N:,}행으로 정렬",
}

# 11_01만 모델 레코드로 저장(exp2 정렬본은 파생물이라 교차 파일에만 남긴다).
result = {**meta, "tag": TARGET["tag"], "arch": TARGET["arch"], "recipe": TARGET["recipe"],
          "data": TARGET["data"], **records[TARGET["tag"]]}
fp = config["out_path"] / f"error_analysis_{TARGET['tag']}.json"
fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved: {fp}")

cleaning_verdict = {
    "question": "정리 데이터 재훈련의 클리닝 성분이 단일 런에서 분리되는가",
    "design": "충돌 연루 74클래스 대 비연루 114클래스의 per-class F1 paired diff-in-diff(레시피 변화는 두 집단에 공통)",
    "n_involved": int(inv.sum()),
    "n_not_involved": int((~inv).sum()),
    "mean_delta_involved_pt": round(100 * float(delta[inv].mean()), 3),
    "mean_delta_not_involved_pt": round(100 * float(delta[~inv].mean()), 3),
    "diff_in_diff_pt": round(100 * obs, 3),
    "permutation_p": round(p_value, 4),
    "n_permutations": PERM,
    "decision": "연루 클래스 초과 이득 있음" if p_value < 0.05 else "분리 불가(노이즈 내)",
    "per_class_delta": {m: round(100 * float(d), 2) for m, d in zip(MNOS, delta)},
}

fp = config["out_path"] / "error_analysis_cleandata_vs_exp2.json"
fp.write_text(json.dumps({**meta, "headline": {t: head[t] for t in TAGS},
                          **cross_model, "cleaning_verdict": cleaning_verdict},
                         ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved: {fp}")

saved: C:\workspace\patent_disc\output\error_analysis_modernbert-patent-len512-b128.json
saved: C:\workspace\patent_disc\output\error_analysis_cleandata_vs_exp2.json
